# World Bank Inflation Data Analysis & Dashboard
## Comprehensive Pipeline: Preprocessing → EDA → Interactive Dashboard

## 1. Environment & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
from pathlib import Path
import sys

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
np.random.seed(42)

print("✓ All libraries imported successfully")

## 2. Load CSVs (World Bank & Merged Data)

In [ ]:
# Define paths
data_dir = Path('../data')
data_dir.mkdir(parents=True, exist_ok=True)

# Load merged inflation data
df_merged = pd.read_csv('../data/merged_inflation_data_20251118.csv')

print("=" * 60)
print("MERGED INFLATION DATA")
print("=" * 60)
print(f"Shape: {df_merged.shape}")
print(f"\nColumns: {df_merged.columns.tolist()}")
print(f"\nData Types:\n{df_merged.dtypes}")
print(f"\nMemory Usage:\n{df_merged.memory_usage(deep=True)}")
print(f"\nFirst 10 rows:")
print(df_merged.head(10))
print(f"\nBasic Statistics:")
print(df_merged.describe())

## 3. Inspect & Validate Raw Schema

In [ ]:
def validate_schema(df, name):
    """Validate required columns and data integrity"""
    print(f"\n{'='*60}")
    print(f"SCHEMA VALIDATION: {name}")
    print(f"{'='*60}")
    
    required_cols = ['country', 'country_code', 'year', 'inflation_rate']
    missing_cols = [col for col in required_cols if col not in df.columns]
    
    if missing_cols:
        print(f"❌ Missing columns: {missing_cols}")
    else:
        print(f"✓ All required columns present")
    
    # Check year range
    print(f"\nYear Range: {df['year'].min()} - {df['year'].max()}")
    print(f"Unique Countries: {df['country'].nunique()}")
    print(f"Unique Country Codes: {df['country_code'].nunique()}")
    
    # Duplicates
    duplicates = df.duplicated().sum()
    print(f"Duplicate Rows: {duplicates}")
    
    # Missing values
    print(f"\nMissing Values:")
    print(df.isnull().sum())
    
    # Inconsistent country codes
    country_code_mismatch = df.groupby('country')['country_code'].nunique()
    mismatches = country_code_mismatch[country_code_mismatch > 1]
    if len(mismatches) > 0:
        print(f"\n⚠ Countries with multiple codes: {len(mismatches)}")
        print(mismatches.head())

validate_schema(df_merged, "Merged Data")

## 4. Type Casting & Basic Cleaning

In [ ]:
def clean_dataframe(df):
    """Perform basic data cleaning and type conversions"""
    df = df.copy()
    
    print("\nCLEANING OPERATIONS:")
    
    # Type conversions
    df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
    df['inflation_rate'] = pd.to_numeric(df['inflation_rate'], errors='coerce').astype('float64')
    print("✓ Type conversions completed")
    
    # Trim whitespace in country names
    if 'country' in df.columns:
        df['country'] = df['country'].str.strip()
    print("✓ Whitespace trimmed")
    
    # Remove rows with missing critical values
    initial_rows = len(df)
    df = df.dropna(subset=['year', 'inflation_rate', 'country_code'])
    removed = initial_rows - len(df)
    print(f"✓ Removed {removed} rows with missing critical values")
    
    return df

df = clean_dataframe(df_merged)
print(f"\nFinal shape after cleaning: {df.shape}")

## 5. Harmonize Country Identifiers

In [ ]:
def harmonize_countries(df):
    """Ensure consistent country codes and names"""
    df = df.copy()
    
    # Create mapping from country_code to country name
    country_mapping = df.drop_duplicates(subset=['country_code']).set_index('country_code')['country'].to_dict()
    
    # Fill missing country names
    df['country'] = df['country_code'].map(country_mapping)
    
    print(f"✓ Country harmonization complete")
    print(f"Total unique countries: {df['country'].nunique()}")
    
    return df

df = harmonize_countries(df)
print(f"\nSample harmonized data:")
print(df[['country', 'country_code', 'year', 'inflation_rate']].head(15))

## 6. Feature Engineering (YoY, Rolling Averages, Anomalies)

In [ ]:
def engineer_features(df):
    """Create derived features for analysis"""
    df = df.copy()
    
    # Sort for time-series operations
    df = df.sort_values(['country_code', 'year']).reset_index(drop=True)
    
    print("\nFEATURE ENGINEERING:")
    
    # Year-over-year change
    df['yoy_change'] = df.groupby('country_code')['inflation_rate'].diff()
    print("✓ Year-over-year change calculated")
    
    # Rolling averages
    df['inflation_3yr_ma'] = df.groupby('country_code')['inflation_rate'].transform(
        lambda x: x.rolling(window=3, center=False).mean()
    )
    df['inflation_5yr_ma'] = df.groupby('country_code')['inflation_rate'].transform(
        lambda x: x.rolling(window=5, center=False).mean()
    )
    print("✓ Rolling averages (3yr, 5yr) calculated")
    
    # Rolling std
    df['inflation_rolling_std'] = df.groupby('country_code')['inflation_rate'].transform(
        lambda x: x.rolling(window=3, center=False).std()
    )
    print("✓ Rolling standard deviation calculated")
    
    # Inflation category
    def categorize(rate):
        if pd.isna(rate):
            return 'Unknown'
        elif rate < 2:
            return 'Low'
        elif rate < 5:
            return 'Moderate'
        elif rate < 10:
            return 'High'
        else:
            return 'Very High'
    
    df['inflation_category'] = df['inflation_rate'].apply(categorize)
    print("✓ Inflation categories assigned")
    
    # Z-score anomaly detection
    df['z_score'] = df.groupby('country_code')['inflation_rate'].transform(
        lambda x: np.abs((x - x.mean()) / x.std())
    )
    df['is_anomaly'] = df['z_score'] > 2.5
    print(f"✓ Anomaly detection complete (flagged: {df['is_anomaly'].sum()} records)")
    
    return df

df = engineer_features(df)
print(f"\nFinal shape with features: {df.shape}")
print(f"\nSample with new features:")
print(df[['country', 'year', 'inflation_rate', 'yoy_change', 'inflation_category', 'is_anomaly']].head(20))

## 7. Exploratory Data Analysis — Summary Statistics

In [ ]:
# Overall statistics
print("\n" + "="*60)
print("OVERALL INFLATION STATISTICS")
print("="*60)
print(df['inflation_rate'].describe())

# By source
print("\n" + "="*60)
print("STATISTICS BY SOURCE")
print("="*60)
if 'source' in df.columns:
    print(df.groupby('source')['inflation_rate'].describe())

# By inflation category
print("\n" + "="*60)
print("DISTRIBUTION BY INFLATION CATEGORY")
print("="*60)
print(df['inflation_category'].value_counts().sort_values(ascending=False))

# Top countries by latest inflation
print("\n" + "="*60)
print("TOP 15 COUNTRIES BY LATEST INFLATION RATE")
print("="*60)
latest_year = df['year'].max()
top_countries = df[df['year'] == latest_year].nlargest(15, 'inflation_rate')[['country', 'inflation_rate', 'year']]
print(top_countries.to_string())

## 8. Exploratory Data Analysis — Visualizations

In [ ]:
# Histogram of inflation rates
fig1 = px.histogram(df, x='inflation_rate', nbins=50, 
                    title='Distribution of Inflation Rates Worldwide',
                    labels={'inflation_rate': 'Inflation Rate (%)', 'count': 'Frequency'})
fig1.show()

# KDE plot
fig2 = go.Figure()
inflation_clean = df['inflation_rate'].dropna()
from scipy import stats
kde = stats.gaussian_kde(inflation_clean[inflation_clean < 50])  # Remove extreme outliers for KDE
x_range = np.linspace(inflation_clean[inflation_clean < 50].min(), 
                       inflation_clean[inflation_clean < 50].max(), 200)
fig2.add_trace(go.Scatter(x=x_range, y=kde(x_range), mode='lines', name='KDE'))
fig2.update_layout(title='Kernel Density Estimate of Inflation Rates (< 50%)',
                   xaxis_title='Inflation Rate (%)', yaxis_title='Density')
fig2.show()

In [ ]:
# Top 10 countries by latest inflation
latest_inflation = df[df['year'] == df['year'].max()].nlargest(10, 'inflation_rate')
fig3 = px.bar(latest_inflation, x='country', y='inflation_rate', 
              title=f'Top 10 Countries by Inflation Rate ({latest_inflation["year"].iloc[0]})',
              labels={'inflation_rate': 'Inflation Rate (%)'})
fig3.update_xaxes(tickangle=-45)
fig3.show()

In [ ]:
# Time series for selected countries
selected_countries = ['United States', 'China', 'India', 'Brazil', 'Argentina', 'Turkey']
df_selected = df[df['country'].isin(selected_countries)].dropna(subset=['inflation_rate'])

fig4 = px.line(df_selected, x='year', y='inflation_rate', color='country',
               title='Inflation Trends for Major Economies',
               labels={'inflation_rate': 'Inflation Rate (%)', 'year': 'Year'})
fig4.show()

In [ ]:
# Inflation category distribution pie chart
category_counts = df[df['year'] == df['year'].max()]['inflation_category'].value_counts()
fig5 = px.pie(values=category_counts.values, names=category_counts.index,
              title=f'Distribution of Countries by Inflation Category ({latest_year})')
fig5.show()

In [ ]:
# Global inflation trends over time
global_avg = df.groupby('year')['inflation_rate'].agg(['mean', 'median', 'std']).reset_index()

fig6 = go.Figure()
fig6.add_trace(go.Scatter(x=global_avg['year'], y=global_avg['mean'], 
                          mode='lines+markers', name='Mean',
                          line=dict(color='blue', width=3)))
fig6.add_trace(go.Scatter(x=global_avg['year'], y=global_avg['median'], 
                          mode='lines+markers', name='Median',
                          line=dict(color='green', width=3)))
fig6.update_layout(title='Global Average Inflation Trends',
                   xaxis_title='Year', yaxis_title='Inflation Rate (%)',
                   hovermode='x unified')
fig6.show()

## 9. Outlier Detection & Anomalies

In [ ]:
# Anomalies detected
anomalies = df[df['is_anomaly'] == True].sort_values('inflation_rate', ascending=False)

print(f"\nTOTAL ANOMALIES DETECTED: {len(anomalies)}")
print(f"\nTop 20 Anomalies:")
print(anomalies[['country', 'year', 'inflation_rate', 'z_score']].head(20).to_string())

# Visualize anomalies
fig7 = px.scatter(df.dropna(subset=['z_score']), x='year', y='inflation_rate', 
                  color='is_anomaly', size='z_score',
                  title='Inflation Anomalies Detection',
                  labels={'is_anomaly': 'Is Anomaly', 'inflation_rate': 'Inflation Rate (%)'},
                  color_discrete_map={True: 'red', False: 'blue'})
fig7.show()

## 10. Save Cleaned Data

In [ ]:
# Save processed data
output_path = Path('../data/processed_inflation_data.csv')
df.to_csv(output_path, index=False)

print(f"\n✓ Cleaned data saved to: {output_path}")
print(f"  Shape: {df.shape}")
print(f"  File size: {output_path.stat().st_size / 1024:.2f} KB")

## 11. Dash + Plotly Dashboard — App Setup

In [ ]:
from dash import Dash, dcc, html, Input, Output, callback
import plotly.express as px
import plotly.graph_objects as go

# Initialize Dash app
app = Dash(__name__)

# Get unique countries and years
countries = sorted(df['country'].unique())
years = sorted(df['year'].unique())

print(f"Dashboard initialized with {len(countries)} countries and {len(years)} years of data")
print(f"Year range: {years[0]} - {years[-1]}")

In [ ]:
# Define app layout
app.layout = html.Div([
    html.Div([
        html.H1("World Bank Inflation Data Analysis Dashboard", 
                style={'textAlign': 'center', 'marginBottom': 30}),
        html.Hr()
    ], style={'padding': '20px', 'backgroundColor': '#f8f9fa', 'marginBottom': 20}),
    
    # Control Panel
    html.Div([
        html.Div([
            html.Label("Select Country:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='country-dropdown',
                options=[{'label': 'All Countries', 'value': 'ALL'}] + 
                        [{'label': c, 'value': c} for c in countries],
                value='United States',
                style={'width': '100%'}
            )
        ], style={'flex': '1', 'marginRight': '20px'}),
        
        html.Div([
            html.Label("Year Range:", style={'fontWeight': 'bold'}),
            dcc.RangeSlider(
                id='year-slider',
                min=years[0],
                max=years[-1],
                value=[years[0], years[-1]],
                marks={year: str(year) for year in range(years[0], years[-1] + 1, 5)},
                tooltip={"placement": "bottom", "always_visible": True}
            )
        ], style={'flex': '2', 'marginLeft': '20px'})
    ], style={'display': 'flex', 'marginBottom': 30, 'backgroundColor': '#fff', 
              'padding': '20px', 'borderRadius': '5px', 'boxShadow': '0 2px 4px rgba(0,0,0,0.1)'}),
    
    # Row 1: Time series and distribution
    html.Div([
        html.Div([dcc.Graph(id='time-series-graph')], 
                style={'flex': '1', 'marginRight': '10px'}),
        html.Div([dcc.Graph(id='distribution-graph')], 
                style={'flex': '1', 'marginLeft': '10px'})
    ], style={'display': 'flex', 'marginBottom': 20}),
    
    # Row 2: Statistics and top countries
    html.Div([
        html.Div([dcc.Graph(id='stats-graph')], 
                style={'flex': '1', 'marginRight': '10px'}),
        html.Div([dcc.Graph(id='top-countries-graph')], 
                style={'flex': '1', 'marginLeft': '10px'})
    ], style={'display': 'flex', 'marginBottom': 20}),
    
    # Row 3: Anomalies and category distribution
    html.Div([
        html.Div([dcc.Graph(id='anomaly-graph')], 
                style={'flex': '1', 'marginRight': '10px'}),
        html.Div([dcc.Graph(id='category-graph')], 
                style={'flex': '1', 'marginLeft': '10px'})
    ], style={'display': 'flex'})
], style={'padding': '20px', 'fontFamily': 'Arial, sans-serif', 'backgroundColor': '#f5f5f5', 'minHeight': '100vh'})

print("✓ Dashboard layout defined")

## 12. Dash Callbacks — Interactive Updates

In [ ]:
@app.callback(
    [
        Output('time-series-graph', 'figure'),
        Output('distribution-graph', 'figure'),
        Output('stats-graph', 'figure'),
        Output('top-countries-graph', 'figure'),
        Output('anomaly-graph', 'figure'),
        Output('category-graph', 'figure')
    ],
    [
        Input('country-dropdown', 'value'),
        Input('year-slider', 'value')
    ]
)
def update_dashboard(selected_country, year_range):
    # Filter data
    filtered_df = df[(df['year'] >= year_range[0]) & (df['year'] <= year_range[1])]
    
    if selected_country != 'ALL':
        country_data = filtered_df[filtered_df['country'] == selected_country]
    else:
        country_data = filtered_df
    
    # 1. Time Series
    if selected_country != 'ALL':
        ts_data = country_data.dropna(subset=['inflation_rate']).sort_values('year')
        fig1 = go.Figure()
        fig1.add_trace(go.Scatter(x=ts_data['year'], y=ts_data['inflation_rate'], 
                                  mode='lines+markers', name=selected_country,
                                  line=dict(color='#1f77b4', width=3)))
        if not ts_data['inflation_3yr_ma'].empty:
            fig1.add_trace(go.Scatter(x=ts_data['year'], y=ts_data['inflation_3yr_ma'], 
                                      mode='lines', name='3-Year MA',
                                      line=dict(color='red', dash='dash')))
        fig1.update_layout(title=f'Inflation Trend - {selected_country}',
                          xaxis_title='Year', yaxis_title='Inflation Rate (%)',
                          hovermode='x unified', height=400)
    else:
        global_trend = filtered_df.groupby('year')['inflation_rate'].mean().reset_index()
        fig1 = px.line(global_trend, x='year', y='inflation_rate',
                      title='Global Average Inflation Trend',
                      labels={'inflation_rate': 'Avg Inflation Rate (%)', 'year': 'Year'},
                      height=400)
    
    # 2. Distribution
    fig2 = px.histogram(country_data.dropna(subset=['inflation_rate']), 
                       x='inflation_rate', nbins=30,
                       title='Inflation Rate Distribution',
                       labels={'inflation_rate': 'Inflation Rate (%)'},
                       height=400)
    
    # 3. Statistics Box
    stats_text = country_data['inflation_rate'].describe()
    fig3 = go.Figure(data=[go.Table(
        header=dict(values=['Statistic', 'Value']),
        cells=dict(values=[
            ['Count', 'Mean', 'Std Dev', 'Min', '25%', '50%', '75%', 'Max'],
            [f"{stats_text['count']:.0f}", f"{stats_text['mean']:.2f}", 
             f"{stats_text['std']:.2f}", f"{stats_text['min']:.2f}",
             f"{stats_text['25%']:.2f}", f"{stats_text['50%']:.2f}",
             f"{stats_text['75%']:.2f}", f"{stats_text['max']:.2f}"]
        ])
    )])
    fig3.update_layout(title='Summary Statistics', height=400, margin=dict(l=0, r=0, t=40, b=0))
    
    # 4. Top Countries
    top_countries_data = filtered_df[filtered_df['year'] == year_range[1]].nlargest(10, 'inflation_rate')
    fig4 = px.bar(top_countries_data, x='country', y='inflation_rate',
                  title=f'Top 10 Countries by Inflation ({year_range[1]})',
                  labels={'inflation_rate': 'Inflation Rate (%)'},
                  height=400)
    fig4.update_xaxes(tickangle=-45)
    
    # 5. Anomalies
    anomaly_data = country_data[country_data['is_anomaly'] == True]
    fig5 = px.scatter(anomaly_data, x='year', y='inflation_rate',
                     title='Detected Inflation Anomalies',
                     labels={'inflation_rate': 'Inflation Rate (%)'},
                     size='z_score', hover_data=['country'],
                     height=400)
    
    # 6. Category Distribution
    category_dist = country_data['inflation_category'].value_counts()
    fig6 = px.pie(values=category_dist.values, names=category_dist.index,
                  title='Distribution by Inflation Category',
                  height=400)
    
    return fig1, fig2, fig3, fig4, fig5, fig6

print("✓ Dashboard callbacks defined")

## 13. Run the Dashboard

In [ ]:
# Run the app (uncomment to run)
# if __name__ == '__main__':
#     app.run_server(debug=True, port=8050)

print("\n" + "="*60)
print("DASHBOARD READY!")
print("="*60)
print("\nTo run the dashboard:")
print("  1. Uncomment and run the cell below")
print("  2. Or run: python -m scripts.run_dashboard")
print("\nAccess dashboard at: http://127.0.0.1:8050/")

## 14. Summary & Export

In [ ]:
print("\n" + "="*80)
print(" " * 20 + "ANALYSIS SUMMARY")
print("="*80)

print(f"\n📊 DATA OVERVIEW:")
print(f"  • Total Records: {len(df):,}")
print(f"  • Countries: {df['country'].nunique()}")
print(f"  • Year Range: {df['year'].min()} - {df['year'].max()}")
print(f"  • Time Period: {df['year'].max() - df['year'].min() + 1} years")

print(f"\n📈 INFLATION STATISTICS:")
print(f"  • Global Mean: {df['inflation_rate'].mean():.2f}%")
print(f"  • Global Median: {df['inflation_rate'].median():.2f}%")
print(f"  • Global Std Dev: {df['inflation_rate'].std():.2f}%")
print(f"  • Min: {df['inflation_rate'].min():.2f}%")
print(f"  • Max: {df['inflation_rate'].max():.2f}%")

print(f"\n🎯 INFLATION CATEGORIES (Latest Year):")
latest_categories = df[df['year'] == df['year'].max()]['inflation_category'].value_counts()
for cat, count in latest_categories.items():
    print(f"  • {cat}: {count} countries")

print(f"\n⚠️  ANOMALIES:")
print(f"  • Total Flagged: {df['is_anomaly'].sum()}")
print(f"  • Percentage: {df['is_anomaly'].sum() / len(df) * 100:.1f}%")

print(f"\n💾 FILES CREATED:")
print(f"  • Processed Data: ../data/processed_inflation_data.csv")
print(f"  • Dashboard Script: ../scripts/run_dashboard.py")
print(f"  • Preprocessing Module: ../scripts/preprocess.py")

print("\n" + "="*80)